# 01 — Data Preparation · California 2020 Wildfires

**Purpose.** Load, inspect, clean, reproject, spatially join, and clip every
input layer, then write analysis-ready intermediates to `data/processed/`.
Notebook `02_analysis_viz.ipynb` reads **only** from `data/processed/`.

**Sections**
- **1.0** Setup — imports, CRS constants, paths, style
- **1.1** Load & inspect — schema discovery (vectors, FVEG `.gdb`, MTBS `.tif`)
- 1.2 Clean DINS structures
- 1.3 Clean fire perimeters
- 1.4 WUI prep
- 1.5 Spatial joins
- 1.6 Raster clips (FVEG + MTBS → buffered perimeters)
- 1.7 Write processed outputs + manifest

> **CRS discipline.** All area / distance / buffer / spatial-join math happens in
> **EPSG:3310** (California Albers). `EPSG:4326` is used for web display only.
> Reproject explicitly and assert CRS before every spatial op.


## 1.0 · Setup

In [1]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

import sys, re, json, subprocess
import xml.etree.ElementTree as ET
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import fiona
import rasterio
import rioxarray  # noqa: F401  (registers .rio accessor)

# --- paths (notebook lives in notebooks/, project root is one level up) ---
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = ROOT / "data" / "raw"
PROC = ROOT / "data" / "processed"
PROC.mkdir(parents=True, exist_ok=True)

# --- shared style module ---
sys.path.insert(0, str(ROOT / "src"))
import style
style.apply_mpl_theme()

# --- CRS constants ---
CRS_EQUAL_AREA = "EPSG:3310"   # California Albers — ALL metric math
CRS_WEB        = "EPSG:4326"   # Folium / Plotly display only

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print("geopandas", gpd.__version__, "| rasterio", rasterio.__version__,
      "| fiona", fiona.__version__)
print("ROOT:", ROOT)
print("RAW :", RAW, "exists:", RAW.exists())
print("PROC:", PROC, "exists:", PROC.exists())

geopandas 1.1.3 | rasterio 1.4.4 | fiona 1.10.1
ROOT: /Users/annamika/Documents/AlbertSchool - MscY2/Env Risk Assessment/California Wildfires/ca-wildfire-2020
RAW : /Users/annamika/Documents/AlbertSchool - MscY2/Env Risk Assessment/California Wildfires/ca-wildfire-2020/data/raw exists: True
PROC: /Users/annamika/Documents/AlbertSchool - MscY2/Env Risk Assessment/California Wildfires/ca-wildfire-2020/data/processed exists: True


## 1.1 · Load & inspect (schema discovery)

We print `.shape`, `.crs`, `.columns` for each vector layer, decode the FVEG
raster attribute table (value → WHR13 habitat class), and read the MTBS `.xml`
sidecar for the burn-severity class definitions. Findings are summarised at the
end of this section so the schema is documented in-notebook.

In [2]:
def banner(t):
    print("\n" + "=" * 74 + "\n" + t + "\n" + "=" * 74)

def inspect(gdf, name):
    print(f"shape={gdf.shape}  crs={gdf.crs}")
    print("geom:", gdf.geom_type.value_counts().to_dict())
    print("columns:", list(gdf.columns))


In [3]:
banner("fire_perimeters_2020.geojson")
perims_raw = gpd.read_file(RAW / "fire_perimeters_2020.geojson")
inspect(perims_raw, "perimeters")
display(perims_raw[["FIRE_NAME", "GIS_ACRES", "ALARM_DATE", "CONT_DATE"]].head())


fire_perimeters_2020.geojson


shape=(504, 21)  crs=EPSG:3857
geom: {'Polygon': 410, 'MultiPolygon': 94}
columns: ['OBJECTID', 'YEAR_', 'STATE', 'AGENCY', 'UNIT_ID', 'FIRE_NAME', 'INC_NUM', 'IRWINID', 'ALARM_DATE', 'CONT_DATE', 'CAUSE', 'C_METHOD', 'OBJECTIVE', 'GIS_ACRES', 'COMMENTS', 'COMPLEX_NAME', 'COMPLEX_ID', 'FIRE_NUM', 'GlobalID', 'DECADES', 'geometry']


,FIRE_NAME,GIS_ACRES,ALARM_DATE,CONT_DATE
0,AUGUST COMPLEX,1032700.0,"Sun, 16 Aug 2020 00:00:00 GMT","Wed, 11 Nov 2020 00:00:00 GMT"
1,SCU LIGHTNING COMPLEX,396824.5,"Sun, 16 Aug 2020 00:00:00 GMT","Fri, 11 Sep 2020 00:00:00 GMT"
2,CREEK,379842.4,"Fri, 04 Sep 2020 00:00:00 GMT","Thu, 24 Dec 2020 00:00:00 GMT"
3,NORTH COMPLEX,318797.3,"Mon, 17 Aug 2020 00:00:00 GMT","Thu, 03 Dec 2020 00:00:00 GMT"
4,HENNESSEY,305351.9,"Mon, 17 Aug 2020 00:00:00 GMT","Wed, 16 Sep 2020 00:00:00 GMT"


In [4]:
banner("dins_2020.geojson  (28,626 inspected structures)")
dins_raw = gpd.read_file(RAW / "dins_2020.geojson")
inspect(dins_raw, "dins")

print("\nDAMAGE value_counts:")
print(dins_raw["DAMAGE"].value_counts(dropna=False))

print("\nSTRUCTURECATEGORY value_counts:")
print(dins_raw["STRUCTURECATEGORY"].value_counts(dropna=False))

v = pd.to_numeric(dins_raw["ASSESSEDIMPROVEDVALUE"], errors="coerce")
print(f"\nASSESSEDIMPROVEDVALUE  non-null={100*v.notna().mean():.1f}%  "
      f">0={100*(v>0).mean():.1f}%  median(>0)=${v[v>0].median():,.0f}")


dins_2020.geojson  (28,626 inspected structures)


shape=(28626, 46)  crs=EPSG:4326
geom: {'Point': 28626}
columns: ['OBJECTID', 'GLOBALID', 'DAMAGE', 'STRUCTURETYPE', 'STREETNUMBER', 'STREETNAME', 'STREETTYPE', 'STREETSUFFIX', 'CITY', 'STATE', 'ZIPCODE', 'CALFIREUNIT', 'COUNTY', 'COMMUNITY', 'BATTALION', 'INCIDENTNAME', 'INCIDENTNUM', 'INCIDENTSTARTDATE', 'HAZARDTYPE', 'WHEREFIRESTARTEDONSTRUCTURE', 'WHATDIDFIRESTARTFROM', 'DEFENSIVEACTIONS', 'STRUCTURECATEGORY', 'NUMBEROFUNITPERSTRUCTURE', 'NOOUTBUILDINGSDAMAGED', 'NOOUTBUILDINGSNOTDAMAGED', 'NOOFCARSONPROPERTY', 'ROOFCONSTRUCTION', 'EAVES', 'VENTSCREEN', 'EXTERIORSIDING', 'WINDOWPANE', 'DECKPORCHONGRADE', 'DECKPORCHELEVATED', 'PATIOCOVERCARPORT', 'FENCEATTACHEDTOSTRUCTURE', 'PROPANETANKDISTANCE', 'UTILITYMISCSTRUCTUREDISTANCE', 'FIRENAME', 'APN', 'ASSESSEDIMPROVEDVALUE', 'YEARBUILT', 'SITEADDRESS', 'LATITUDE', 'LONGITUDE', 'geometry']

DAMAGE value_counts:
DAMAGE
No Damage            17801
Destroyed (>50%)      9494
Affected (>0-10%)      826
Inaccessible           204
Minor (10-25%

In [5]:
banner("ca_counties_2020.geojson")
counties_raw = gpd.read_file(RAW / "ca_counties_2020.geojson")
inspect(counties_raw, "counties")
display(counties_raw[["NAME", "GEOID"]].head())


ca_counties_2020.geojson
shape=(58, 13)  crs=EPSG:4269
geom: {'Polygon': 52, 'MultiPolygon': 6}
columns: ['STATEFP', 'COUNTYFP', 'COUNTYNS', 'AFFGEOID', 'GEOID', 'NAME', 'NAMELSAD', 'STUSPS', 'STATE_NAME', 'LSAD', 'ALAND', 'AWATER', 'geometry']


,NAME,GEOID
0,Alameda,06001
1,San Mateo,06081
2,Riverside,06065
3,Solano,06095
4,Tehama,06103


In [6]:
banner("wui_california.geojson  (read schema only — 222 MB file)")
# Peek at columns from a small slice, then get class tallies via OGR SQL
wui_head = gpd.read_file(RAW / "wui_california.geojson", rows=50)
print("columns:", list(wui_head.columns))
print("\nWUI class field = WUI_DESC ; hazard field = HAZ_DESC")
print("WUI_DESC sample values:", sorted(wui_head["WUI_DESC"].dropna().unique()))

# Full class tally without loading the whole file into memory
sql = ("SELECT WUI_DESC, COUNT(*) c FROM Wildland_Urban_Interface "
       "GROUP BY WUI_DESC ORDER BY c DESC")
out = subprocess.run(
    ["ogrinfo", "-dialect", "SQLITE", "-sql", sql,
     str(RAW / "wui_california.geojson")],
    capture_output=True, text=True).stdout
print("\nWUI_DESC class counts (full dataset):")
print("\n".join(l for l in out.splitlines()
                 if "WUI_DESC (String)" in l or "c (Integer)" in l))


wui_california.geojson  (read schema only — 222 MB file)


columns: ['OBJECTID', 'WUI_NUM', 'HAZ_NUM', 'DEN4', 'WUI_DESC', 'HAZ_DESC', 'SHAPELEN', 'SHAPEAREA', 'geometry']

WUI class field = WUI_DESC ; hazard field = HAZ_DESC
WUI_DESC sample values: ['Influence Zone', 'Interface', 'Intermix']



WUI_DESC class counts (full dataset):
  WUI_DESC (String) = Intermix
  c (Integer) = 16884
  WUI_DESC (String) = Influence Zone
  c (Integer) = 11767
  WUI_DESC (String) = Interface
  c (Integer) = 7506


### FVEG vegetation raster (`fveg22_1.gdb`)

The FVEG file geodatabase stores a **single-band categorical raster** (not a
vector layer — `fiona.listlayers` fails on it). It opens directly with rasterio.
The pixel value → habitat-class mapping lives in an embedded **Raster Attribute
Table (RAT)**, which we decode from `gdalinfo`'s XML output. The 13-class
lifeform field **`WHR13NAME`** is what Act 2 tabulates.

In [7]:
banner("FVEG raster: fveg22_1.gdb")
with rasterio.open(RAW / "fveg22_1.gdb") as ds:
    print(f"crs={ds.crs}  size={ds.width}x{ds.height}  res={ds.res}  "
          f"dtype={ds.dtypes[0]}  nodata={ds.nodata}")
    print(f"bounds={ds.bounds}")

# --- decode the Raster Attribute Table from gdalinfo XML ---
def load_fveg_rat(gdb_path):
    txt = subprocess.run(["gdalinfo", str(gdb_path)],
                         capture_output=True, text=True).stdout
    m = re.search(r"<GDALRasterAttributeTable.*?</GDALRasterAttributeTable>",
                  txt, re.S)
    rat = ET.fromstring(m.group(0))
    cols = [fd.find("Name").text for fd in rat.findall("FieldDefn")]
    rows = [[f.text for f in r.findall("F")] for r in rat.findall("Row")]
    df = pd.DataFrame(rows, columns=cols)
    df["Value"] = df["Value"].astype(int)
    df["Count"] = df["Count"].astype(float)
    return df

fveg_rat = load_fveg_rat(RAW / "fveg22_1.gdb")
print(f"\nRAT shape: {fveg_rat.shape}  (one row per distinct pixel value)")
print("RAT fields:", list(fveg_rat.columns))
print("\nWHR13 habitat classes (statewide pixel totals):")
display(fveg_rat.groupby("WHR13NAME")["Count"].sum()
        .sort_values(ascending=False).to_frame("statewide_px"))


FVEG raster: fveg22_1.gdb
crs=EPSG:3310  size=39623x38094  res=(30.0, 30.0)  dtype=uint16  nodata=None
bounds=BoundingBox(left=-636269.9999999981, bottom=-612510.0, right=552420.0000000019, top=530310.0)



RAT shape: (42032, 15)  (one row per distinct pixel value)
RAT fields: ['Value', 'Count', 'WHRALL', 'LIFEFORM', 'WHRNUM', 'WHRNAME', 'WHRTYPE', 'WHRSIZE', 'WHRDENSITY', 'WHR10NUM', 'WHR10NAME', 'WHR13NUM', 'WHR13NAME', 'SOURCE_NAME', 'SOURCE_YEAR']

WHR13 habitat classes (statewide pixel totals):


,statewide_px
WHR13NAME,
Desert Shrub,96851228.0
Conifer Forest,79525851.0
Shrub,62218348.0
Herbaceous,52048638.0
Agriculture,49084571.0
Hardwood Woodland,27867414.0
Hardwood Forest,24107260.0
Urban,23387703.0
Conifer Woodland,15332595.0


### MTBS burn-severity mosaic (`mtbs_CA_2020.tif`)

Categorical `uint8`, **EPSG:5070** (CONUS Albers), 30 m, `nodata=0`. Class
definitions are read from the `.xml` sidecar (`edomv` → `edomvd`).

In [8]:
banner("MTBS raster: mtbs_CA_2020.tif")
mtbs_path = RAW / "mtbs_CA_2020" / "mtbs_CA_2020.tif"
mt = rioxarray.open_rasterio(mtbs_path)
print(f"crs={mt.rio.crs}  shape={tuple(mt.shape)}  res={mt.rio.resolution()}  "
      f"dtype={mt.dtype}  nodata={mt.rio.nodata}")
print(f"bounds={mt.rio.bounds()}")

vals, counts = np.unique(mt.values, return_counts=True)

# class definitions from the .xml sidecar
xml = (mtbs_path.with_suffix(".xml")).read_text(errors="ignore")
edoms = re.findall(r"<edomv>(\d+)</edomv>\s*<edomvd>(.*?)</edomvd>", xml, re.S)
defs = {int(k): v.strip() for k, v in edoms}

print("\nMTBS severity classes (statewide pixel counts):")
display(pd.DataFrame({
    "value": vals,
    "definition": [defs.get(int(v), "?") for v in vals],
    "statewide_px": counts,
}))


MTBS raster: mtbs_CA_2020.tif
crs=EPSG:5070  shape=(1, 39289, 20086)  res=(30.0, -30.0)  dtype=uint8  nodata=0
bounds=(-2321445.0, 1276095.0, -1718865.0, 2454765.0)



MTBS severity classes (statewide pixel counts):


,value,definition,statewide_px
0,0,Background/No Data,771414693
1,1,Unburned/Underburned to Low Burn Severity,2582660
2,2,Low burn severity,6924832
3,3,Moderate burn severity,4903024
4,4,High Burn Severity,3276997
5,5,Increased Greenness/Increased Vegetation Response,40661
6,6,Non-Processing Area Mask,15987


### 1.1 Schema-discovery summary (documented in-notebook)

| Layer | Rows | Native CRS | Key fields |
|---|---|---|---|
| `fire_perimeters_2020` | 504 | **EPSG:3857** | `FIRE_NAME`, `GIS_ACRES`, `ALARM_DATE`, `CONT_DATE`, `COMPLEX_NAME` |
| `dins_2020` | 28,626 pts | **EPSG:4326** | `DAMAGE`, `ASSESSEDIMPROVEDVALUE`, `STRUCTURETYPE`, `STRUCTURECATEGORY`, `COUNTY`, `FIRENAME`, `APN`, construction features |
| `ca_counties_2020` | 58 | **EPSG:4269** | `NAME`, `GEOID` |
| `wui_california` | 36,157 | **EPSG:4326** | **`WUI_DESC`** ∈ {Interface, Intermix, Influence Zone}; `HAZ_DESC` |
| `fveg22_1.gdb` | raster 39623×38094 | **EPSG:3310** | single band uint16; RAT → **`WHR13NAME`** (13 habitat classes) |
| `mtbs_CA_2020.tif` | raster 20086×39289 | **EPSG:5070** | uint8, nodata=0; severity 1–6 |

**DINS `DAMAGE` classes** (confirmed): `No Damage` (17,801), `Destroyed (>50%)`
(9,494), `Affected (>0-10%)` (826), `Inaccessible` (204), `Minor (10-25%)`
(194), `Major (25-50%)` (107). Value coverage: **90.5 %** non-null,
**80.6 %** positive; median positive assessed value ≈ **\$199,962**.

**MTBS severity classes** (from `.xml`): `0` Background/No Data · `1`
Unburned/Low · `2` Low · `3` Moderate · `4` High · `5` Increased Greenness ·
`6` Non-Processing Mask.

**FVEG `WHR13NAME`** (13 classes): Conifer Forest, Conifer Woodland, Hardwood
Forest, Hardwood Woodland, Shrub, Desert Shrub, Herbaceous, Agriculture, Urban,
Barren/Other, Water, Desert Woodland, Wetland.

> **Convenient.** FVEG is already in EPSG:3310 (our equal-area CRS). MTBS is in
> EPSG:5070, so perimeters get reprojected to 5070 for the MTBS clip and to 3310
> for the FVEG clip (section 1.6).

---
*Next: section 1.2 (clean DINS) onward — built after schema review is approved.*


## 1.2 · Clean DINS structures

- Normalise `DAMAGE` to an **ordered categorical** (`style.DAMAGE_ORDER`).
- Coerce `ASSESSEDIMPROVEDVALUE` → numeric; flag value coverage.
- Build a `damage_fraction` (rule 4a): Destroyed=1.0, Major=0.375, Minor=0.175,
  Affected=0.05, No Damage=0, **Inaccessible=NaN** (Act 4 shows it both ways —
  as destroyed and excluded).
- **Geometry vs LAT/LON check:** the provided EPSG:4326 geometry is preferred;
  we confirm it agrees with the `LATITUDE`/`LONGITUDE` columns to ~5 dp and only
  fall back to lat/lon where a valid geometry is missing.

In [9]:
from pandas.api.types import CategoricalDtype

dins = dins_raw.copy()

# --- geometry vs lat/lon agreement (prefer geometry, fall back to lat/lon) ---
lon = pd.to_numeric(dins["LONGITUDE"], errors="coerce")
lat = pd.to_numeric(dins["LATITUDE"], errors="coerce")
has_geom = dins.geometry.notna() & ~dins.geometry.is_empty
gx, gy = dins.geometry.x, dins.geometry.y
disagree = has_geom & ((gx - lon).abs() > 1e-5) | has_geom & ((gy - lat).abs() > 1e-5)
print(f"rows with valid geometry : {has_geom.sum():,} / {len(dins):,}")
print(f"geometry disagrees with lat/lon (>1e-5 deg): {int(disagree.sum())}")
print(f"rows missing geometry (will fall back to lat/lon): {int((~has_geom).sum())}")

# fall back to lat/lon ONLY where geometry is missing/empty
if (~has_geom).any():
    fill_pts = gpd.points_from_xy(lon[~has_geom], lat[~has_geom])
    dins.loc[~has_geom, "geometry"] = fill_pts
print("-> trusting provided geometry; lat/lon used only as fallback / sanity check")

rows with valid geometry : 28,626 / 28,626
geometry disagrees with lat/lon (>1e-5 deg): 15
rows missing geometry (will fall back to lat/lon): 0
-> trusting provided geometry; lat/lon used only as fallback / sanity check


In [10]:
# --- ordered DAMAGE categorical ---
dmg_dtype = CategoricalDtype(categories=style.DAMAGE_ORDER, ordered=True)
dins["DAMAGE"] = dins["DAMAGE"].astype(dmg_dtype)
assert dins["DAMAGE"].isna().sum() == 0, "unmapped DAMAGE labels present!"

# --- damage_fraction (rule 4a); Inaccessible left NaN for dual treatment ---
FRAC = {"No Damage": 0.0, "Affected (>0-10%)": 0.05, "Minor (10-25%)": 0.175,
        "Major (25-50%)": 0.375, "Destroyed (>50%)": 1.0, "Inaccessible": np.nan}
dins["damage_fraction"] = dins["DAMAGE"].astype(str).map(FRAC).astype(float)

# --- assessed value + coverage flags ---
dins["assessed_value"] = pd.to_numeric(dins["ASSESSEDIMPROVEDVALUE"], errors="coerce")
dins["value_nonnull"] = dins["assessed_value"].notna()
dins["value_positive"] = dins["assessed_value"].fillna(0) > 0
dins["year_built"] = pd.to_numeric(dins["YEARBUILT"], errors="coerce")

print("value coverage: %.1f%% non-null, %.1f%% positive"
      % (100*dins["value_nonnull"].mean(), 100*dins["value_positive"].mean()))
print("\ndamage_fraction by class:")
print(dins.groupby("DAMAGE", observed=True)["damage_fraction"].first())

# --- reproject to equal-area ---
dins_3310 = dins.to_crs(CRS_EQUAL_AREA)
assert dins_3310.crs.to_epsg() == 3310
print("\ndins_3310:", dins_3310.shape, dins_3310.crs)

value coverage: 90.5% non-null, 80.6% positive

damage_fraction by class:
DAMAGE
No Damage            0.000
Affected (>0-10%)    0.050
Minor (10-25%)       0.175
Major (25-50%)       0.375
Destroyed (>50%)     1.000
Inaccessible           NaN
Name: damage_fraction, dtype: float64

dins_3310: (28626, 51) EPSG:3310


## 1.3 · Clean fire perimeters

Reproject to 3310, repair geometries (`make_valid`), compute area in acres from
the geometry and cross-check against `GIS_ACRES`, and parse `ALARM_DATE` /
`CONT_DATE` to datetimes for the time-lapse.

In [11]:
ACRE_M2 = 4046.8564224

perims = perims_raw.to_crs(CRS_EQUAL_AREA).copy()
perims["geometry"] = perims.geometry.make_valid()
assert perims.crs.to_epsg() == 3310

perims["acres_geom"] = perims.geometry.area / ACRE_M2
perims["GIS_ACRES"] = pd.to_numeric(perims["GIS_ACRES"], errors="coerce")
perims["alarm_date"] = pd.to_datetime(perims["ALARM_DATE"], errors="coerce", utc=True)
perims["cont_date"]  = pd.to_datetime(perims["CONT_DATE"],  errors="coerce", utc=True)

# cross-check geometry acres vs reported GIS_ACRES
ok = perims["GIS_ACRES"] > 0
ratio = (perims.loc[ok, "acres_geom"] / perims.loc[ok, "GIS_ACRES"])
print(f"acres_geom vs GIS_ACRES  median ratio={ratio.median():.3f}  "
      f"(1.0 = perfect)   corr={perims.loc[ok,'acres_geom'].corr(perims.loc[ok,'GIS_ACRES']):.4f}")
print(f"total acres (geometry) = {perims['acres_geom'].sum():,.0f}")
print(f"alarm_date parsed: {perims['alarm_date'].notna().sum()}/{len(perims)}  "
      f"range {perims['alarm_date'].min().date()} -> {perims['alarm_date'].max().date()}")
print("\nLargest 5 fires by geometry acres:")
print(perims.nlargest(5, "acres_geom")[["FIRE_NAME", "acres_geom", "GIS_ACRES"]]
      .to_string(index=False))

acres_geom vs GIS_ACRES  median ratio=1.000  (1.0 = perfect)   corr=1.0000
total acres (geometry) = 4,182,280
alarm_date parsed: 504/504  range 2020-01-21 -> 2020-12-31

Largest 5 fires by geometry acres:
            FIRE_NAME   acres_geom  GIS_ACRES
       AUGUST COMPLEX 1.032700e+06  1032700.0
SCU LIGHTNING COMPLEX 3.968227e+05   396824.5
                CREEK 3.798457e+05   379842.4
        NORTH COMPLEX 3.187987e+05   318797.3
            HENNESSEY 3.053493e+05   305351.9


## 1.4 · WUI prep

Reproject to 3310, repair, and **dissolve by `WUI_DESC`** (Interface / Intermix /
Influence Zone) for Act 3 area math and mapping. The full (undissolved) layer is
kept in memory for the point-in-polygon structure join in §1.5.

> **Caveat (carried into narrative):** the WUI layer is a county-level pattern
> dataset built on recent housing vintage — it is *not* 2020-exact and is not
> suitable for single-parcel adjudication.

In [12]:
wui = gpd.read_file(RAW / "wui_california.geojson")
print("loaded WUI:", wui.shape, wui.crs)
wui_3310 = wui.to_crs(CRS_EQUAL_AREA)
wui_3310["geometry"] = wui_3310.geometry.make_valid()

wui_diss = (wui_3310[["WUI_DESC", "geometry"]]
            .dissolve(by="WUI_DESC", as_index=False))
wui_diss["acres"] = wui_diss.geometry.area / ACRE_M2
assert wui_3310.crs.to_epsg() == 3310 and wui_diss.crs.to_epsg() == 3310
print("\ndissolved WUI by class (statewide acres):")
print(wui_diss[["WUI_DESC", "acres"]].to_string(index=False))

loaded WUI: (36157, 9) EPSG:4326



dissolved WUI by class (statewide acres):
      WUI_DESC        acres
Influence Zone 1.463541e+07
     Interface 1.059469e+06
      Intermix 2.530616e+06


## 1.5 · Spatial joins (all in EPSG:3310)

DINS structures → **county**, → **WUI class**, → **fire perimeter**. We print
join coverage and cross-check the spatial perimeter match against the DINS
`FIRENAME` field, reporting the mismatch count (names use different conventions —
e.g. complex vs sub-fire — so a non-zero count is expected and quantified).

In [13]:
def first_match(left_geom, right, cols):
    """sjoin within, keep first match per left row, return the chosen columns."""
    j = gpd.sjoin(left_geom, right[cols + ["geometry"]], how="left", predicate="within")
    j = j[~j.index.duplicated(keep="first")]
    return j[cols]

pts = dins_3310[["geometry"]]

cty = counties_raw.to_crs(CRS_EQUAL_AREA)[["NAME", "GEOID", "geometry"]]
m_cty = first_match(pts, cty.rename(columns={"NAME": "county_sj", "GEOID": "geoid_sj"}),
                    ["county_sj", "geoid_sj"])

m_wui = first_match(pts, wui_3310.rename(columns={"WUI_DESC": "wui_class"}), ["wui_class"])

m_per = first_match(pts, perims.rename(columns={"FIRE_NAME": "perim_fire"}), ["perim_fire"])

dins_3310 = dins_3310.join(m_cty).join(m_wui).join(m_per)

n = len(dins_3310)
cov_cty = 100 * dins_3310["county_sj"].notna().mean()
cov_wui = 100 * dins_3310["wui_class"].notna().mean()
cov_per = 100 * dins_3310["perim_fire"].notna().mean()
print("JOIN COVERAGE (of %d DINS structures):" % n)
print(f"  -> county     : {cov_cty:5.1f}%")
print(f"  -> WUI class   : {cov_wui:5.1f}%   (structures outside any WUI polygon are non-WUI)")
print(f"  -> fire perim  : {cov_per:5.1f}%")
print("\nWUI class distribution of DINS structures:")
print(dins_3310["wui_class"].value_counts(dropna=False))

JOIN COVERAGE (of 28626 DINS structures):
  -> county     : 100.0%
  -> WUI class   :  85.2%   (structures outside any WUI polygon are non-WUI)
  -> fire perim  :  74.5%

WUI class distribution of DINS structures:
wui_class
Intermix          14479
Influence Zone     6343
NaN                4223
Interface          3581
Name: count, dtype: int64


In [14]:
# --- FIRENAME (attribute) vs spatial-join perimeter cross-check ---
a = dins_3310["FIRENAME"].astype("string").str.upper().str.strip()
b = dins_3310["perim_fire"].astype("string").str.upper().str.strip()
both = a.notna() & b.notna()
mismatch = int((both & (a != b)).sum())
print(f"FIRENAME vs spatial-join perimeter:")
print(f"  both present : {int(both.sum()):,}")
print(f"  mismatches   : {mismatch:,}  ({100*mismatch/max(int(both.sum()),1):.1f}% of comparable)")
print("\nTop mismatched (DINS FIRENAME -> spatial perim_fire) pairs:")
mm = (pd.DataFrame({"FIRENAME": a[both & (a != b)], "perim_fire": b[both & (a != b)]})
      .value_counts().head(10))
print(mm.to_string())

FIRENAME vs spatial-join perimeter:
  both present : 16,261
  mismatches   : 7,208  (44.3% of comparable)

Top mismatched (DINS FIRENAME -> spatial perim_fire) pairs:
FIRENAME              perim_fire    
BEAR                  NORTH COMPLEX     3134
NAPA                  GLASS             2239
SONOMA                GLASS             1128
SANTA ROSA            GLASS              336
AUGUST COMPLEX WEST   AUGUST COMPLEX     111
AUGUST COMPLEX SOUTH  AUGUST COMPLEX      97
ST. HELENA            GLASS               72
ELKHORN               AUGUST COMPLEX      40
DOE                   AUGUST COMPLEX      25
QUAIL                 HENNESSEY           16


## 1.6 · Raster clips & zonal tabulation (clip-FIRST)

Dissolve perimeters → buffer **100 m** → explode into **non-overlapping parts**
(correct statewide totals, no double counting, memory-safe windowed reads).

**MTBS severity semantics (refinement — baked in here):**
- Classes **1–4** = the burn-severity story (Unburned/Low, Low, Moderate, High).
- Class **5** (Increased Greenness) is tabulated **separately**, never inside the
  burn-severity stack.
- Classes **0** (background/nodata) and **6** (non-processing mask) are **excluded
  entirely** from the burned denominator.

We also quantify **MTBS perimeter coverage** (#2): how many of the 504 perimeters
contain any burned (1–4) pixel, and what share of total burned-perimeter acres
those covered fires represent.

In [15]:
import rasterio.mask as rmask
PIX_ACRES = 900 / ACRE_M2   # 30 m x 30 m pixel -> acres

# dissolved, buffered, exploded into non-overlapping parts (in 3310)
mask_union = perims.geometry.buffer(100).union_all()
parts_3310 = gpd.GeoSeries(mask_union, crs=CRS_EQUAL_AREA).explode(index_parts=False)
parts_3310 = parts_3310.reset_index(drop=True)
print(f"dissolved+buffered mask -> {len(parts_3310)} non-overlapping parts")
print(f"mask area = {parts_3310.area.sum()/ACRE_M2:,.0f} acres "
      f"(vs sum of individual perimeters {perims['acres_geom'].sum():,.0f} "
      "-> difference = perimeter overlap removed)")

dissolved+buffered mask -> 556 non-overlapping parts
mask area = 4,412,395 acres (vs sum of individual perimeters 4,182,280 -> difference = perimeter overlap removed)


In [16]:
# ---- FVEG (EPSG:3310): acres of each WHR13 habitat class within perimeters ----
whr_classes = sorted(fveg_rat["WHR13NAME"].dropna().unique())
code_of = {c: i for i, c in enumerate(whr_classes)}
maxv = int(fveg_rat["Value"].max())
lut = np.full(maxv + 1, -1, dtype=np.int32)
for v, nm in zip(fveg_rat["Value"].astype(int), fveg_rat["WHR13NAME"]):
    lut[v] = code_of[nm]

whr_counts = np.zeros(len(whr_classes), dtype=np.int64)
with rasterio.open(RAW / "fveg22_1.gdb") as fds:
    for geom in parts_3310.geometry:
        try:
            out, _ = rmask.mask(fds, [geom], crop=True, filled=True, nodata=0)
        except ValueError:
            continue  # part outside raster extent (edge sliver) — skip
        d = out[0].ravel()
        d = d[(d > 0) & (d <= maxv)]
        cc = lut[d]
        cc = cc[cc >= 0]
        whr_counts += np.bincount(cc, minlength=len(whr_classes))

fveg_tab = (pd.DataFrame({"WHR13NAME": whr_classes,
                          "pixels": whr_counts,
                          "acres": whr_counts * PIX_ACRES})
            .sort_values("acres", ascending=False).reset_index(drop=True))
print("FVEG habitat acres within 2020 fire perimeters (top 13):")
print(fveg_tab.to_string(index=False, float_format=lambda x: f"{x:,.0f}"))

FVEG habitat acres within 2020 fire perimeters (top 13):
        WHR13NAME  pixels     acres
  Hardwood Forest 4979215 1,107,352
            Shrub 4973568 1,106,096
   Conifer Forest 4739277 1,053,991
       Herbaceous 1958363   435,530
Hardwood Woodland 1859017   413,436
 Conifer Woodland  297290    66,116
  Desert Woodland  207573    46,163
     Barren/Other  163540    36,370
      Agriculture   90446    20,115
            Water   75281    16,742
     Desert Shrub   71014    15,793
            Urban   55091    12,252
          Wetland   40126     8,924


In [17]:
# ---- MTBS (EPSG:5070): statewide severity acres via non-overlapping parts ----
mtbs_path = RAW / "mtbs_CA_2020" / "mtbs_CA_2020.tif"
parts_5070 = parts_3310.to_crs("EPSG:5070")

sev_counts = np.zeros(7, dtype=np.int64)
with rasterio.open(mtbs_path) as mds:
    for geom in parts_5070.geometry:
        try:
            out, _ = rmask.mask(mds, [geom], crop=True, filled=True, nodata=0)
        except ValueError:
            continue  # part outside raster extent — skip
        sev_counts += np.bincount(out.ravel(), minlength=7)
sev_counts[0] = 0  # 0 = background/nodata, excluded

_, _, sev_labels = style.severity_listed_cmap()
mtbs_tab = pd.DataFrame({
    "value": list(range(1, 7)),
    "label": [sev_labels[i] for i in range(1, 7)],
    "pixels": sev_counts[1:7],
    "acres": sev_counts[1:7] * PIX_ACRES,
})
# semantics flags
mtbs_tab["role"] = ["burn_severity"]*4 + ["greenness", "mask_excluded"]
burned_acres = mtbs_tab.loc[mtbs_tab["role"] == "burn_severity", "acres"].sum()
print("MTBS severity within 2020 fire perimeters:")
print(mtbs_tab.to_string(index=False, float_format=lambda x: f"{x:,.0f}"))
print(f"\n-> BURNED landscape (classes 1-4 only) = {burned_acres:,.0f} acres")
print(f"-> Increased Greenness (class 5, separate) = "
      f"{mtbs_tab.loc[mtbs_tab.value==5,'acres'].iloc[0]:,.0f} acres")
print("-> classes 0 & 6 excluded from burned denominator")

MTBS severity within 2020 fire perimeters:
 value               label  pixels     acres          role
     1      Unburned / Low 2323120   516,650 burn_severity
     2                 Low 6728891 1,496,471 burn_severity
     3            Moderate 4800464 1,067,598 burn_severity
     4                High 3274299   728,187 burn_severity
     5 Increased Greenness   39483     8,781     greenness
     6 Non-Processing Mask   14719     3,273 mask_excluded

-> BURNED landscape (classes 1-4 only) = 3,808,906 acres
-> Increased Greenness (class 5, separate) = 8,781 acres
-> classes 0 & 6 excluded from burned denominator


In [18]:
# ---- MTBS perimeter-coverage check (refinement #2) + per-fire severity ----
perims_buf_5070 = perims.geometry.buffer(100).to_crs("EPSG:5070")
covered = pd.Series(False, index=perims.index)
perfire = []
with rasterio.open(mtbs_path) as mds:
    for idx, geom in perims_buf_5070.items():
        try:
            out, _ = rmask.mask(mds, [geom], crop=True, filled=True, nodata=0)
        except ValueError:
            # geometry entirely outside raster extent
            continue
        bc = np.bincount(out.ravel(), minlength=7); bc[0] = 0
        burned = int(bc[1:5].sum())
        covered[idx] = burned > 0
        perfire.append({"FIRE_NAME": perims.loc[idx, "FIRE_NAME"],
                        "acres_geom": perims.loc[idx, "acres_geom"],
                        **{f"sev{i}_acres": bc[i]*PIX_ACRES for i in range(1, 7)},
                        "burned_acres": burned*PIX_ACRES})
mtbs_by_fire = pd.DataFrame(perfire)

X = int(covered.sum())
Y = 100 * perims.loc[covered, "acres_geom"].sum() / perims["acres_geom"].sum()
print(f"MTBS COVERAGE: {X} of {len(perims)} perimeters contain >=1 burned (1-4) "
      f"pixel,\n   representing {Y:.1f}% of total burned (perimeter) acres.")
print(f"   -> {len(perims)-X} small perimeters have no MTBS mapping "
      "(expected: below the ~1,000-acre MTBS threshold).")

MTBS COVERAGE: 91 of 504 perimeters contain >=1 burned (1-4) pixel,
   representing 91.3% of total burned (perimeter) acres.
   -> 413 small perimeters have no MTBS mapping (expected: below the ~1,000-acre MTBS threshold).


## 1.7 · Write processed outputs + manifest

Everything notebook 02 needs is written to `data/processed/`. Vectors are
written as GeoParquet (fast, typed); zonal tabulations as CSV; the clipped MTBS
mosaic as a compressed GeoTIFF for the Act 2 severity map.

In [19]:
def save_vec(gdf, name):
    p = PROC / f"{name}.parquet"
    try:
        gdf.to_parquet(p)
    except Exception as e:
        p = PROC / f"{name}.geojson"
        gdf.to_file(p, driver="GeoJSON")
        print(f"   (parquet failed for {name}: {e}; wrote GeoJSON)")
    return p

written = []
# vectors
written.append(save_vec(dins_3310, "dins_clean_3310"))
written.append(save_vec(perims[["FIRE_NAME", "COMPLEX_NAME", "CAUSE", "GIS_ACRES",
                                 "acres_geom", "alarm_date", "cont_date", "geometry"]],
                        "perims_clean_3310"))
written.append(save_vec(wui_diss, "wui_dissolved_3310"))
written.append(save_vec(counties_raw.to_crs(CRS_EQUAL_AREA)[["NAME", "GEOID", "geometry"]],
                        "counties_3310"))
# tabulations
for df, nm in [(fveg_tab, "fveg_whr13_burned"),
               (mtbs_tab, "mtbs_severity_burned"),
               (mtbs_by_fire, "mtbs_severity_by_fire")]:
    p = PROC / f"{nm}.csv"; df.to_csv(p, index=False); written.append(p)

In [20]:
# clipped MTBS mosaic (masked + cropped to perimeter union; deflate-compressed)
with rasterio.open(mtbs_path) as mds:
    out, tr = rmask.mask(mds, list(parts_5070.geometry), crop=True,
                         filled=True, nodata=0)
    prof = mds.profile.copy()
    prof.update(height=out.shape[1], width=out.shape[2], transform=tr,
                compress="deflate", nodata=0)
    p = PROC / "mtbs_clipped_perims.tif"
    with rasterio.open(p, "w", **prof) as dst:
        dst.write(out)
written.append(p)
print("clipped MTBS:", out.shape, "-> non-zero pixels:", int((out > 0).sum()))

clipped MTBS: (1, 39106, 20086) -> non-zero pixels: 17180976


In [21]:
print("=" * 70)
print("PROCESSED MANIFEST  (data/processed/)")
print("=" * 70)
for p in written:
    mb = p.stat().st_size / 1e6
    print(f"  {p.name:32s} {mb:8.1f} MB")

print("\n" + "=" * 70)
print("DATA-QUALITY CHECKPOINTS")
print("=" * 70)
print(f"join coverage  county={cov_cty:.1f}%  WUI={cov_wui:.1f}%  perim={cov_per:.1f}%")
print(f"FIRENAME vs spatial mismatch = {mismatch:,} of {int(both.sum()):,} comparable")
print(f"MTBS coverage = {X}/{len(perims)} perimeters = {Y:.1f}% of burned acres")
print("\nNotebook 01 complete -> data/processed/ populated for notebook 02.")

PROCESSED MANIFEST  (data/processed/)
  dins_clean_3310.parquet               3.3 MB
  perims_clean_3310.parquet            10.8 MB
  wui_dissolved_3310.parquet           63.2 MB
  counties_3310.parquet                 0.7 MB
  fveg_whr13_burned.csv                 0.0 MB
  mtbs_severity_burned.csv              0.0 MB
  mtbs_severity_by_fire.csv             0.0 MB
  mtbs_clipped_perims.tif               4.3 MB

DATA-QUALITY CHECKPOINTS
join coverage  county=100.0%  WUI=85.2%  perim=74.5%
FIRENAME vs spatial mismatch = 7,208 of 16,261 comparable
MTBS coverage = 91/504 perimeters = 91.3% of burned acres

Notebook 01 complete -> data/processed/ populated for notebook 02.
